In [1]:
from os import cpu_count
from qiskit_aer import AerSimulator
from qiskit import*
from tqdm import tqdm
import numpy as np
from numpy import pi,sin,cos
from numpy.random import randint,choice
from qiskit.quantum_info import Statevector
from multiprocessing import Pool
from datetime import datetime
from sklearn.metrics import r2_score as r2
import pandas as pd

In [2]:
def parallel(vector:list,func):
    with Pool(int(cpu_count())) as p:
        return p.map(func,vector)

## Quantum Circuits Functions

In [ ]:
def circuit_singlet_creator(angle_between_analyzers:float):
    """
        ----------------------------------------------
        ----- REMEBER TO MODIFY THIS ALPHA VALUE -----
        ----------------------------------------------
    """
    global_shift = 1*pi/8 # alpha angle between analizer 0 and 1 for alice, angle between x and analizer 0 for bob
    # ---------------------------------------
    q = QuantumCircuit(2,2)
    # Noise indroduction based on QST experiments on QD XX-X-0 cascade
    #state = choice(a=[0,1,2,3,4],p=[0.910,0.003,0.014,0.028,0.045]) # review probabilities
    state = 0
    if state == 0:
        q.h(0); q.rz(relative_phase,0); q.cx(0,1) # singlet state with phase
    elif state == 1:
        q.barrier()#q.z(0);q.z(1) # |00>
    elif state == 2:
        q.x(1); # |01>
    elif state == 3:
        q.x(0) # |10>
    elif state == 4:
        q.x(1);q.x(1) # |11>
    # Basis selection
    
    basis_alice = choice([0,1,2]); basis_bob = choice([1,2,3])
    basis_list = {0:0,1:global_shift,2:(global_shift+angle_between_analyzers),3:3*pi/8}
    q.ry(basis_list[basis_alice],0); q.ry(basis_list[basis_bob],1)
    # Measure
    q.measure(0,0); q.measure(1,1)
    basis = str(basis_alice)+str(basis_bob)
    
    return q,basis
        

In [5]:
def circuit_list_vector_creator(n_states:int,angle_between_analyzers:float):
    circs_basis_vector = parallel([angle_between_analyzers for k in range(n_states)],circuit_singlet_creator)
    return circs_basis_vector

In [6]:
def exec(q:QuantumCircuit):
    simulator = AerSimulator()
    circ = transpile(q, simulator)
    result = simulator.run(circ,shots=1).result().get_counts()
    return list(dict(result).keys())[0]

## Post Execution Functions

In [7]:
def info_selector(circs_basis_vector:list):
    N = len(circs_basis_vector)
    circuits = list()
    basis=list()
    for k in range(N):
        circuits.append(circs_basis_vector[k][0])
        basis.append(circs_basis_vector[k][1])
    return [circuits,basis]

def probability_computation(basis_vector:list,result_vector:list):
    N_same_basis = 0
    N_anticorrelation = 0
    N_states = len(basis_vector)
    for k in range(N_states):
        if basis_vector[k] == '11' or basis_vector[k] == '22':
            N_same_basis += 1
            if result_vector[k] == '00' or result_vector[k] == '11':
                N_anticorrelation += 1
            else:
                pass
        else:
            pass
    return N_anticorrelation/N_same_basis

## Run Function

In [8]:
def local_run (n_states:int,angle_1_2:float):
    info = circuit_list_vector_creator(n_states=n_states,angle_between_analyzers=angle_1_2)
    circuit_list, basis_list = info_selector(info)
    del info
    result = parallel(vector=circuit_list,func=exec)
    result = [result[k]for k in range(len(result))]
    del circuit_list
    fidelity = probability_computation(basis_vector=basis_list,result_vector=result)    
    return fidelity

# Analytical Model

In [9]:
def analytical(theta:float,beta:float,alpha:float):
    return (\
        cos(alpha)**4 + sin(alpha)**4 +\
        2 * (sin(alpha)**2) * (cos(alpha)**2) * cos(theta) + \
        cos(alpha+beta)**4 + sin(alpha+beta)**4 + \
        2 * (sin(alpha+beta)**2) * (cos(alpha+beta)**2) * cos(theta)\
        )/2  

In [10]:
def correlation_computing(n_execs:int,n_phase:int,experiment:np.ndarray):
    theta = np.linspace(0,2*pi,n_phase)
    beta = np.linspace(0,2*pi,n_execs)
    """
        ----------------------------------------------
        ----- REMEBER TO MODIFY THIS ALPHA VALUE -----
        ----------------------------------------------
    """
    alpha = 1*pi/8 
    
    theoretical = np.array([[analytical(theta=t,beta=b, alpha=alpha) for t in theta] for b in beta])
    
    print(f"theoretical = ↓\n{theoretical}\n\nexperimental = ↓\n{experiment}")

    return  r2(y_true=theoretical,y_pred=experiment)
    

In [11]:
def main(n_states:int,n_execs:int):
    fidelity = list()
    for k in range(n_execs):
        fidelity.append(local_run(n_states=n_states,angle_1_2=angle_between_analyzers[k]))
    #print(f"correlation R2 = {correlation_computing}")
    return np.array(fidelity)
    
def save_info_to_csv(column:list,backend_name,time_str):
    file_name = f"E91_data_{backend_name}_{time_str}.csv"
    string = ""
    for data in column:
        string += f"{data},"
    with open(file_name, "a") as file:
        file.write(string + "\n")

In [12]:
states = int(20000) # total states created by the source in the QKD Run 
execs = int(50) # steps of angle between oincident analyzers beta parameter between 0 and 2 pi
n_phase = int(50) # steps of FSS parameter theta between 0 and 2 pi

phase = np.linspace(0,2*pi,n_phase)

global angle_between_analyzers
angle_between_analyzers = np.linspace(0,2*pi,execs)

time_now = datetime.now().strftime("%d_%m_%Y_%H_%M_%S")

for rel_phase in tqdm(phase,desc="progress: "):
    global relative_phase
    relative_phase=rel_phase # phase between bell pair state
    #---------------------------------------------------
    out = main(n_states=states,n_execs=execs)
    #if rel_phase != 0.:
    #    save_info_to_csv(column=[rel_phase,out],backend_name="2026_aer_simulator",time_str=time_now)
    #else:
        #save_info_to_csv(column=['#',angle_between_analyzers],backend_name="2026_aer_simulator",time_str=time_now)
    save_info_to_csv(column=out,backend_name="2026_aer_simulator",time_str=time_now)
    #---------------------------------------------------

progress:   0%|                                                                                                            | 0/50 [00:00<?, ?it/s]Process ForkPoolWorker-24:
Process ForkPoolWorker-15:
Process ForkPoolWorker-19:
Process ForkPoolWorker-22:
Process ForkPoolWorker-21:
Process ForkPoolWorker-14:
Process ForkPoolWorker-17:
Process ForkPoolWorker-13:
Process ForkPoolWorker-23:
Process ForkPoolWorker-18:
Process ForkPoolWorker-20:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/qadrian/APPS/miniconda3/envs/qiskit/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/qadrian/APPS/miniconda3/envs/qiskit/lib/python3.10/multiprocessing/process.py", line 314, in _b

KeyboardInterrupt: 